In [4]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

local_converter = conversion.Converter('local')
local_converter+=pandas2ri.converter
# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

In [8]:
#assigns node types. 'c' for continuous, 'b' for binary
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

In [9]:
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    node_type = infer_type(df_clean)

    with conversion.localconverter(local_converter):
        Y_r = conversion.py2rpy(df_clean)

    ro.globalenv["Y"] = Y_r
    ro.globalenv["node_type"] = node_type
    ro.globalenv["seed"] = seed

    ro.r('''
    set.seed(seed)
    temp <- dagbagM::hc(
      Y = Y,
      nodeType = node_type,
      whiteList = NULL,
      blackList = NULL,
      tol = 1e-6,
      standardize = TRUE,
      maxStep = 1000,
      restart = 10,
      verbose = FALSE
    )
    adj_mat <- temp$adjacency
    ''')

    with conversion.localconverter(local_converter):
        adjacency = conversion.rpy2py(ro.r('adj_mat'))

    return np.asarray(adjacency), list(df_clean.columns)